In [8]:
import torch
import torchvision.models as models
import numpy as np
import cv2
import time
import tensorrt as trt
import sys

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50()
model.fc = torch.nn.Linear(model.fc.in_features, 6)

state_dict = torch.load("best_model_strategy_2.pth", map_location=device)
model.load_state_dict(state_dict)
model.eval().to(device)
print("PyTorch model loaded successfully!")

dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_file = "cnn_garbage_best_model.onnx"
torch.onnx.export(
    model, dummy_input, onnx_file,
    export_params=True,
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"ONNX model saved at {onnx_file}")

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
trt_file = "cnn_garbage_best_model.trt"

with trt.Builder(TRT_LOGGER) as builder, \
     builder.create_network(flags=1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)) as network, \
     trt.OnnxParser(network, TRT_LOGGER) as parser:

    with open(onnx_file, "rb") as f:
        if not parser.parse(f.read()):
            print("Failed to parse ONNX model:")
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            sys.exit(1)
    print("ONNX model parsed successfully!")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 28)

    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    input_tensor = network.get_input(0)
    profile = builder.create_optimization_profile()
    profile.set_shape(input_tensor.name, min=(1, 3, 224, 224),
                                        opt=(1, 3, 224, 224),
                                        max=(4, 3, 224, 224))
    config.add_optimization_profile(profile)

    serialized_engine = builder.build_serialized_network(network, config)
    if serialized_engine is None:
        raise RuntimeError("TensorRT engine build failed!")

    with open(trt_file, "wb") as f:
        f.write(serialized_engine)
    print(f"TensorRT engine saved at {trt_file}")

def preprocess_image(img_path, size=(224,224)):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Image not found: {img_path}. Skipping.")
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size)
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    return img

image_path = "example1.jpg"
input_image = preprocess_image(image_path)
if input_image is None:
    input_image = np.random.rand(1,3,224,224).astype(np.float32)

def benchmark_pytorch(model, input_tensor, runs=100, device="cpu"):
    input_tensor = torch.tensor(input_tensor).to(device)
    model.to(device)
    torch.cuda.synchronize() if device=="cuda" else None
    start = time.time()
    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)
    torch.cuda.synchronize() if device=="cuda" else None
    return (time.time() - start)/runs

cpu_time = benchmark_pytorch(model, input_image, device="cpu")
gpu_time = benchmark_pytorch(model, input_image, device="cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch CPU latency per image: {cpu_time*1000:.2f} ms")
print(f"PyTorch GPU latency per image: {gpu_time*1000:.2f} ms")

print("TensorRT engine built and saved. You can now run inference using TensorRT runtime APIs.")


PyTorch model loaded successfully!


/tmp/ipykernel_2157614/1268477273.py:21: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


ONNX model saved at cnn_garbage_best_model.onnx
ONNX model parsed successfully!
TensorRT engine saved at cnn_garbage_best_model.trt


[ WARN:0@1243.020] global loadsave.cpp:275 findDecoder imread_('example1.jpg'): can't open/read file: check file path/integrity


PyTorch CPU latency per image: 37.90 ms
PyTorch GPU latency per image: 4.11 ms
TensorRT engine built and saved. You can now run inference using TensorRT runtime APIs.
